In [1]:
import os
import json

In [2]:
root_json = "/mnt/ssd/datasets/deepfake/dataset_json"
os.chdir(root_json)

In [3]:
dataset_name = "200k_live_face_dataset"

In [4]:
path_json = os.path.join(root_json,dataset_name+".json")

In [5]:
print(sorted(os.listdir("/mnt/ssd/datasets/deepfake/dataset_json")))

['200k_live_face_dataset.json', 'Celeb-DF-v2.json', 'data_deepfake_bcad.json', 'data_deepfake_bcad_rev.json', 'deepfake_in_the_wild_test.json', 'df40_train.json', 'df40_train_fs_reduced.json', 'df40_train_fs_rev.json', 'df40_train_fsonly_rev.json', 'df40_train_rev.json', 'ensemble_tuning.json', 'evaluation_indonesian_deepfake_dataset_v2_updated.json', 'facebook_dfdc_test.json', 'facebook_dfdc_test_rev.json', 'facebook_dfdc_train.json', 'facebook_dfdc_train_reduced.json', 'facebook_dfdc_train_reduced_rev.json', 'facebook_dfdc_train_rev.json', 'faceforensics++_test.json', 'faceforensics++_train.json', 'faceforensics++_val.json', 'ffhq.json', 'inswapper.json', 'inswapper_rev.json', 'inswapper_train.json', 'reswapper_test.json', 'reswapper_train.json', 'reswapper_v2.json', 'reswapper_v2_test.json', 'reswapper_v2_train.json', 'reswapper_v2_val.json', 'reswapper_val.json', 'training_indonesian_deepfake_dataset_v2_updated.json']


In [6]:
with open(path_json, 'r') as file:
    data = json.load(file)

In [7]:
list_keys = list(data.keys())

In [8]:
path_img = data[list_keys[4]]["processed_path"]
print(path_img)

/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/face/186513_verify_4ff2e9b4-0edc-4a54-8431-9ea9931dc19e_33e66bd6-0507-4b66-bf11-efa24e7b8cf4_original.jpg.jpg


In [9]:
#list_keys

#### Profile Prediction

In [26]:
from face_detection import FaceDetection
import age_detection as ad
import cv2

fd = FaceDetection() # face detection wrapper

ad_m = ad.AgeClassification()

/home/a.nugroho/miniforge3/envs/verihubs_env_orig/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/a.nugroho/miniforge3/envs/verihubs_env_orig/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/a.nugroho/miniforge3/envs/verihubs_env_orig/lib/python3.9/site-packages/age_detection/main.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data w

In [28]:
# read sample image from file
img = cv2.imread(path_img)

# get face detection result
dets, angle = fd.predict(img)

# Age
out = fd.align_single_face(img,dets,angle,outcolor="RGB",crop_size=300)
img_aligned = out[0]

print(ad_m.predict(img_aligned))
# Gender

20-29


#### Image Quality

In [13]:
from face_detection import FaceDetection
from face_quality_assessment import FaceQualityAssessment 
import cv2

fd = FaceDetection() # face detection wrapper
qa = FaceQualityAssessment()


/home/a.nugroho/miniforge3/envs/verihubs_env_orig/lib/python3.9/site-packages/openvino/runtime/__init__.py:10: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  warnings.warn(


In [14]:
# read sample image from file
img = cv2.imread(path_img)

# get face detection result
dets, angle = fd.predict(img)

# get crop_single_face with loose_factor=1.25
img_cropped, _ = fd.crop_single_face(img, dets, angle, loose_factor = 1.25)

# get blur score
blur_score = qa.blur.blur_score_selfie(img_cropped)

# get blur detection
is_blur = qa.blur.blur_detection(blur_score)

# get dark score
dark_score = qa.dark.contrast_score_selfie(img_cropped)

# get dark detection
is_dark = qa.dark.contrast_detection(dark_score)

# get grayscale score
raw_score = qa.grayscale.grayscale_score_selfie_raw(img_cropped)

# get the grayscale detection
is_gray = qa.grayscale.grayscale_detection(raw_score)

# get the blur score
blur_score = qa.image_quality.quality_score_selfie(img)

# get the blur detection
is_blur = qa.image_quality.blur_detection(blur_score)

print(f"{is_blur}, {is_dark},{is_gray}, {is_blur}")


False, False,False, False


#### Deepfake Smoothness

In [20]:
from face_deepfake import DeepfakeDetection
fdd = DeepfakeDetection()

In [21]:
# crop the image
img_cropped_df, bbox = fd.crop_single_face(
    img, dets, angle, loose_factor=1.3, crop_size=None,square=True
)

In [24]:
# get deepfake score
deepfake_score = fdd.predict(img_cropped_df)

# get deepfake decision
is_deepfake = fdd.classify_predictions(deepfake_score)

print(f"{is_deepfake}")


False


In [38]:
# read sample image from file
img = cv2.imread(path_img)
print(img.shape)
path_img

(256, 256, 3)


'/mnt/ssd/datasets/deepfake/200k_live_face_dataset/processes/live/face/186513_verify_4ff2e9b4-0edc-4a54-8431-9ea9931dc19e_33e66bd6-0507-4b66-bf11-efa24e7b8cf4_original.jpg.jpg'

In [ ]:
# load the image 
img = cv2.imread("samples/blur.jpg")

# get the blur score
blur_score = qa.image_quality.quality_score_selfie(img)

# get the blur detection
is_blur = qa.image_quality.blur_detection(blur_score)